# First Attempt

In [1]:
import torch
import numpy as np

## Load/ Preprocess Data

In [ ]:
import pandas as pd
import numpy as np
import wfdb
import ast

#provided function to load data with information in the agg_df (modified to return a numpy array)
def load_raw_data(df, sampling_rate, path):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(path+f) for f in df.filename_lr]
    else:
        data = [wfdb.rdsamp(path+f) for f in df.filename_hr]
    data = np.array([signal for signal, meta in data]).astype(np.float32)
    return data

# path to dta and selected sampling rate
path = './ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1/'
sampling_rate=100

# load and convert annotation data
Y = pd.read_csv(path+'ptbxl_database.csv', index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x)) #changes class distribution to dict

# Load scp_statements.csv for diagnostic aggregation
agg_df = pd.read_csv(path+'scp_statements.csv', index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1] #only take diagnostic scps

classes = agg_df.index.tolist()

# encodes the labeled as a 44d vector of 1's and zeros (based off of classes list)
def encode_multilabel(class_list):
    return np.array([1 if cls in class_list else 0 for cls in classes])

def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            if y_dic[key] > 50:
                tmp.append(key) #diagnostic, diagnostic_subclass #agg_df.loc[key].index
    return list(set(tmp))

# Apply diagnostic superclass and encoding multilabel
Y['diagnostic_superclass'] = Y.scp_codes.apply(aggregate_diagnostic).apply(encode_multilabel)




In [73]:
agg_df

,description,diagnostic,form,rhythm,diagnostic_class,diagnostic_subclass,Statement Category,SCP-ECG Statement Description,AHA code,aECG REFID,CDISC Code,DICOM Code
NDT,non-diagnostic T abnormalities,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,non-diagnostic T abnormalities,NaN,NaN,NaN,NaN
NST_,non-specific ST changes,1.0,1.0,NaN,STTC,NST_,Basic roots for coding ST-T changes and abnorm...,non-specific ST changes,145.0,MDC_ECG_RHY_STHILOST,NaN,NaN
DIG,digitalis-effect,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,suggests digitalis-effect,205.0,NaN,NaN,NaN
LNGQT,long QT-interval,1.0,1.0,NaN,STTC,STTC,other ST-T descriptive statements,long QT-interval,148.0,NaN,NaN,NaN
NORM,normal ECG,1.0,NaN,NaN,NORM,NORM,Normal/abnormal,normal ECG,1.0,NaN,NaN,F-000B7
IMI,inferior myocardial infarction,1.0,NaN,NaN,MI,IMI,Myocardial Infarction,inferior myocardial infarction,161.0,NaN,NaN,NaN
ASMI,anteroseptal myocardial infarction,1.0,NaN,NaN,MI,AMI,Myocardial Infarction,anteroseptal myocardial infarction,165.0,NaN,NaN,NaN
LVH,left ventricular hypertrophy,1.0,NaN,NaN,HYP,LVH,Ventricular Hypertrophy,left ventricular hypertrophy,142.0,NaN,C71076,NaN
LAFB,left anterior fascicular block,1.0,NaN,NaN,CD,LAFB/LPFB,Intraventricular and intra-atrial Conduction d...,left anterior fascicular block,101.0,MDC_ECG_BEAT_BLK_ANT_L_HEMI,C62267,D3-33140
ISC_,non-specific ischemic,1.0,NaN,NaN,STTC,ISC_,Basic roots for coding ST-T changes and abnorm...,ischemic ST-T changes,226.0,NaN,NaN,NaN


In [64]:
# Further example code 
"""
Y_s = Y.head(10).copy()

# Load raw signal data
X = load_raw_data(Y_s, sampling_rate, path)

# Split data into train and test
test_fold = 10
# Train
X_train = X[np.where(Y_s.strat_fold != test_fold)]
y_train = Y_s[(Y_s.strat_fold != test_fold)].diagnostic_superclass
# Test
X_test = X[np.where(Y_s.strat_fold == test_fold)]
y_test = Y_s[Y_s.strat_fold == test_fold].diagnostic_superclass"""

'\nY_s = Y.head(10).copy()\n\n# Load raw signal data\nX = load_raw_data(Y_s, sampling_rate, path)\n\n# Split data into train and test\ntest_fold = 10\n# Train\nX_train = X[np.where(Y_s.strat_fold != test_fold)]\ny_train = Y_s[(Y_s.strat_fold != test_fold)].diagnostic_superclass\n# Test\nX_test = X[np.where(Y_s.strat_fold == test_fold)]\ny_test = Y_s[Y_s.strat_fold == test_fold].diagnostic_superclass'

In [ ]:
(Y["diagnostic_superclass"].apply(tuple).value_counts())

diagnostic_superclass
(0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)    8623
(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)    2746
(1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)    1439
(0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)     603
(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)     495
                                                                                                                                        ... 
(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)

In [77]:
len(Y)

21837

In [76]:
sum(list(Y["diagnostic_superclass"]))

array([1824,  538,  134,  113, 8946, 1167, 1595, 1176, 1607, 1144, 1102,
        783,  782,  529,  538,  536,  342,  322,  123,  206,  171,  197,
         58,  124,  174,  154,  136,  122,   56,   13,   83,   15,   71,
         74,   31,   39,   29,   18,   17,   14,    4,   15,   15,   14])

In [ ]:
#Y_s.diagnostic_superclass.to_numpy() #example line run of multilabel distribution
#Y['scp_codes']

array([array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
              0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
       array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
              0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
       array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
              0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
       array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
              0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
       array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
              0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
       array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
              0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
       array([0, 0, 0, 0, 1,

In [ ]:
Y['scp_codes']

ecg_id
1                 {'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}
2                             {'NORM': 80.0, 'SBRAD': 0.0}
3                               {'NORM': 100.0, 'SR': 0.0}
4                               {'NORM': 100.0, 'SR': 0.0}
5                               {'NORM': 100.0, 'SR': 0.0}
                               ...                        
21833    {'NDT': 100.0, 'PVC': 100.0, 'VCLVH': 0.0, 'ST...
21834             {'NORM': 100.0, 'ABQRS': 0.0, 'SR': 0.0}
21835                           {'ISCAS': 50.0, 'SR': 0.0}
21836                           {'NORM': 100.0, 'SR': 0.0}
21837                           {'NORM': 100.0, 'SR': 0.0}
Name: scp_codes, Length: 21837, dtype: object

## Data Prep

In [116]:
from sklearn.model_selection import train_test_split

Y_s = Y.sample(200)

Y_transformed = np.stack(Y_s.diagnostic_superclass.values).astype(np.float32)

# Load raw signal data
X = load_raw_data(Y_s, sampling_rate, path)

X = np.transpose(X, (0, 2, 1))

X_train, X_test, y_train, y_test = train_test_split(X, Y_transformed, test_size=0.1, random_state=56)

In [117]:
X_train_torch = torch.tensor(X_train) # change dim ordering for PyTorch
y_train_torch = torch.tensor(y_train)
X_test_torch = torch.tensor(X_test) # change dim ordering for PyTorch
y_test_torch = torch.tensor(y_test)

In [118]:
train_dataset = torch.utils.data.TensorDataset(X_train_torch, y_train_torch)
train_dataloader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)

test_dataset = torch.utils.data.TensorDataset(X_test_torch, y_test_torch)
test_dataloader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=32)


## CNN

In [119]:
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F


class CNet(nn.Module):
  def __init__(self):
    super(CNet,self).__init__()
    #starting with 96

    self.conv11 = nn.Conv1d(12, 64, 9, padding=4)
    self.conv12 = nn.Conv1d(64, 64, 9, padding=4)
    self.bn1 = nn.BatchNorm1d(64)

    self.pool = nn.MaxPool1d(kernel_size=2, stride = 2)

    self.conv21 = nn.Conv1d(64, 128, 9, padding=4)
    self.conv22 = nn.Conv1d(128, 128, 9, padding=4)
    self.bn2 = nn.BatchNorm1d(128)

    self.conv31 = nn.Conv1d(128, 256, 9, padding=4)
    self.conv32 = nn.Conv1d(256, 256, 9, padding=4)
    self.conv33 = nn.Conv1d(256, 256, 9, padding=4)
    self.bn3 = nn.BatchNorm1d(256)

    self.conv41 = nn.Conv1d(256, 512, 9, padding=4)
    self.conv42 = nn.Conv1d(512, 512, 9, padding=4)
    self.conv43 = nn.Conv1d(512, 512, 9, padding=4)
    self.bn4 = nn.BatchNorm1d(512)

    self.fc1 = nn.Linear(512 * 125, 1000)
    self.fc2 = nn.Linear(1000, 500)
    self.fc3 = nn.Linear(500, 44)

  def forward(self, x):
    #x = self.pool(F.relu(self.conv12(F.relu(self.conv11(x)))))
    #x = self.pool(F.relu(self.conv22(F.relu(self.conv21(x)))))
    #x = self.pool(F.relu(self.conv33(F.relu(self.conv32(F.relu(self.conv31(x)))))))
    #x = F.relu(self.conv43(F.relu(self.conv42(F.relu(self.conv41(x))))))

    x = self.pool(F.relu(self.bn1(self.conv12(F.relu(self.conv11(x))))))
    x = self.pool(F.relu(self.bn2(self.conv22(F.relu(self.conv21(x))))))
    x = self.pool(F.relu(self.bn3(self.conv33(F.relu(self.conv32(F.relu(self.conv31(x))))))))
    x = F.relu(self.bn4(self.conv43(F.relu(self.conv42(F.relu(self.conv41(x)))))))
    x = torch.flatten(x,1)
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = self.fc3(x)
    return x

In [120]:
def train_model(model, train_dataloader, test_dataloader, epochs, weight_decay):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    model.train()

    #loss_fn = torch.nn.CrossEntropyLoss()

    pos_weight = torch.tensor([5.0])
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    lr = 1e-3
    opt = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)
    #opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    #scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    best_test = 0
    best_test_train = 0
    best_epoch = 0

    for epoch in range(epochs):
        running_loss = 0
        for batch_index, (X, y) in enumerate(train_dataloader):
            print(f"Epoch: {epoch} Batch {batch_index}")
            X,y = X.to(device), y.to(device)

            opt.zero_grad() #zero out the gradients

            z = model(X)
            loss = loss_fn(z,y) #compute loss
            running_loss += loss.item()

            loss.backward() #compute gradients

            opt.step() #apply gradients
            #scheduler.step()



        running_loss = running_loss / len(train_dataloader)
        print(f"Epoch {epoch} train loss: {running_loss}")


In [37]:
X.shape

(100, 12, 1000)

In [121]:
net = CNet()
train_model(net, train_dataloader, test_dataloader, 30, 1e-2)

Epoch: 0 Batch 0
Epoch: 0 Batch 1
Epoch: 0 Batch 2
Epoch: 0 Batch 3
Epoch: 0 Batch 4
Epoch: 0 Batch 5
Epoch 0 train loss: 0.7618493338425955
Epoch: 1 Batch 0
Epoch: 1 Batch 1
Epoch: 1 Batch 2
Epoch: 1 Batch 3
Epoch: 1 Batch 4
Epoch: 1 Batch 5
Epoch 1 train loss: 0.7579955061276754
Epoch: 2 Batch 0
Epoch: 2 Batch 1
Epoch: 2 Batch 2
Epoch: 2 Batch 3
Epoch: 2 Batch 4
Epoch: 2 Batch 5
Epoch 2 train loss: 0.7535731196403503
Epoch: 3 Batch 0
Epoch: 3 Batch 1
Epoch: 3 Batch 2
Epoch: 3 Batch 3
Epoch: 3 Batch 4
Epoch: 3 Batch 5
Epoch 3 train loss: 0.7494133114814758
Epoch: 4 Batch 0
Epoch: 4 Batch 1
Epoch: 4 Batch 2
Epoch: 4 Batch 3
Epoch: 4 Batch 4
Epoch: 4 Batch 5
Epoch 4 train loss: 0.7459658682346344
Epoch: 5 Batch 0
Epoch: 5 Batch 1
Epoch: 5 Batch 2
Epoch: 5 Batch 3
Epoch: 5 Batch 4
Epoch: 5 Batch 5
Epoch 5 train loss: 0.7420237759749094
Epoch: 6 Batch 0
Epoch: 6 Batch 1
Epoch: 6 Batch 2
Epoch: 6 Batch 3
Epoch: 6 Batch 4
Epoch: 6 Batch 5
Epoch 6 train loss: 0.7384125888347626
Epoch: 7 Batc

In [122]:
def dig_accurate(y_preds, y_ground):
    element_wise_eq = y_preds == y_ground
    print(f"Row-wise Accuracy (perfect match): {sum(torch.all(element_wise_eq, dim=1).int())/element_wise_eq.shape[0]}")
    print(f"Element Wise Accuracy: {sum(element_wise_eq.int().flatten(0))/torch.numel(element_wise_eq)}")

In [123]:
dig_accurate((torch.sigmoid(net(X_test_torch)) > 0.5).int(), y_test_torch.int())

Row-wise Accuracy (perfect match): 0.4000000059604645
Element Wise Accuracy: 0.9670454263687134


In [100]:
print(net(X_test_torch))
print((torch.sigmoid(net(X_test_torch)) > 0.5).int())
print(y_test_torch.int())
print(y_test_torch.shape)
print((torch.sigmoid(net(X_test_torch)) > 0.5).int() == y_test_torch.int()) #== torch.ones(10,44)
print(torch.eq(torch.sigmoid(net(X_test_torch) > 0.5).int(), y_test_torch.int()))


tensor([[ 2.2877e-03, -1.0041e-01, -5.6426e-02, -2.3783e-02,  2.8802e-01,
          5.7111e-02,  1.1516e-01,  2.1411e-01, -7.9751e-03,  3.8361e-01,
          2.6507e-01,  3.5754e-01,  5.2251e-02,  2.0182e-02, -2.5831e-02,
          3.6782e-01, -4.6466e-02, -3.3759e-01, -1.8443e-01, -1.2943e-01,
         -1.6291e-01, -9.6938e-02, -1.3433e-01, -6.0273e-02, -2.4587e-01,
         -1.1776e-01, -1.0455e-01, -4.1973e-01, -4.7630e-01, -3.0209e-01,
         -1.0072e-01, -1.9476e-02, -4.0918e-02, -1.9064e-01,  2.1194e-01,
         -8.6886e-02, -2.3089e-01,  1.4761e-02, -4.2404e-03,  9.7487e-02,
         -2.2746e-01,  1.6353e-01, -5.1663e-02, -3.2609e-01],
        [-1.6693e-01, -2.0723e-01, -1.0561e-01, -1.7320e-01,  2.6169e-01,
          2.0757e-01,  2.9955e-01,  2.0939e-01, -1.9201e-01,  1.7794e-01,
          1.6023e-01,  2.8691e-01,  1.1374e-01,  8.1874e-03,  2.1182e-01,
          3.2571e-01, -9.7735e-02, -4.5011e-01, -2.3396e-01, -1.0202e-01,
         -1.3907e-01, -1.3404e-01, -1.2546e-01,  1